In [1]:
!pip install -q transformers datasets sentencepiece sacrebleu
!pip install -q nltk
!pip install -q rouge-score


In [2]:
from google.colab import files
uploaded = files.upload()

Saving fr-tr.txt.zip to fr-tr.txt (5).zip


In [3]:
import zipfile
import os

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")

os.listdir("data")

['TED2020.fr-tr.fr',
 'TED2020.fr-tr.tr',
 'LICENSE',
 'README',
 'TED2020.fr-tr.xml']

In [4]:
from datasets import Dataset

def load_parallel_dataset(src_file, tgt_file, limit=5000):
    with open(src_file, encoding="utf-8") as f:
        src = [line.strip() for line in f][:limit]
    with open(tgt_file, encoding="utf-8") as f:
        tgt = [line.strip() for line in f][:limit]

    return Dataset.from_dict({
        "source": src,
        "target": tgt
    })


dataset = load_parallel_dataset(
    "data/TED2020.fr-tr.fr",
    "data/TED2020.fr-tr.tr",
    limit=5000
)

dataset


Dataset({
    features: ['source', 'target'],
    num_rows: 5000
})

In [5]:
from transformers import MT5ForConditionalGeneration, MT5TokenizerFast

model_name = "google/mt5-small"

tokenizer = MT5TokenizerFast.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
The tokenizer class 

In [6]:
def preprocess(batch):
    inputs = [
        f"translate French to Turkish: {s}"
        for s in batch["source"]
    ]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        max_length=64
    )

    labels = tokenizer(
        batch["target"],
        truncation=True,
        max_length=64
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [7]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# USE THE MODEL YOU ALREADY HAVE
model.to(device)
model.eval()

def translate(texts, max_length=64):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length
        )

    return tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mt5-fr-tr",
    learning_rate=3e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
    fp16=False,
    torch_compile=False,
)


In [9]:
from transformers import Trainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)


/tmp/ipython-input-847717712.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
sample_src = dataset["source"][:500]
sample_ref = dataset["target"][:500]

baseline_preds = translate(sample_src)

for i in range(5):
    print(f"🔹 Example {i+1}")
    print("FR :", sample_src[i])
    print("REF:", sample_ref[i])
    print("PRED (before):", baseline_preds[i])
    print("-" * 60)


🔹 Example 1
FR : Merci beaucoup, Chris.
REF: Çok teşekkür ederim Chris.
PRED (before): <extra_id_0>
------------------------------------------------------------
🔹 Example 2
FR : C'est vraiment un honneur de pouvoir venir sur cette scène une deuxième fois. Je suis très reconnaissant.
REF: Bu sahnede ikinci kez yer alma fırsatına sahip olmak gerçekten büyük bir onur. Çok minnettarım.
PRED (before): <extra_id_0>. <extra_id_10>.
------------------------------------------------------------
🔹 Example 3
FR : J'ai été très impressionné par cette conférence, et je tiens à vous remercier tous pour vos nombreux et sympathiques commentaires sur ce que j'ai dit l'autre soir.
REF: Bu konferansta çok mutlu oldum, ve anlattıklarımla ilgili güzel yorumlarınız için sizlere çok teşekkür ederim.
PRED (before): <extra_id_0>.
------------------------------------------------------------
🔹 Example 4
FR : Et je dis çà sincèrement, en autres parce que --Faux sanglot-- j'en ai besoin !
REF: Bunu içtenlikle söylü

In [11]:
trainer.train()
optim="adamw_torch"


Step,Training Loss
50,15.670800
100,7.773600
150,6.370800
200,5.932000
250,5.685900
300,5.439500
350,5.240200
400,5.354200
450,5.227900
500,5.022800


In [12]:
trained_preds = translate(sample_src)

for i in range(5):
    print(f"🟢 Example {i+1}")
    print("FR :", sample_src[i])
    print("REF:", sample_ref[i])
    print("PRED (after):", trained_preds[i])
    print("-" * 60)


🟢 Example 1
FR : Merci beaucoup, Chris.
REF: Çok teşekkür ederim Chris.
PRED (after): Bu Chris Chris Chris. Chris.
------------------------------------------------------------
🟢 Example 2
FR : C'est vraiment un honneur de pouvoir venir sur cette scène une deuxième fois. Je suis très reconnaissant.
REF: Bu sahnede ikinci kez yer alma fırsatına sahip olmak gerçekten büyük bir onur. Çok minnettarım.
PRED (after): Bu üç hafta önce iletişim iletişim almak çok teşekkür ederim.
------------------------------------------------------------
🟢 Example 3
FR : J'ai été très impressionné par cette conférence, et je tiens à vous remercier tous pour vos nombreux et sympathiques commentaires sur ce que j'ai dit l'autre soir.
REF: Bu konferansta çok mutlu oldum, ve anlattıklarımla ilgili güzel yorumlarınız için sizlere çok teşekkür ederim.
PRED (after): Bu o gece olarak, onunla iletişim etmek için çok teşekkür ettik, çok çok çok mutlu olduğunu,
-----------------------------------------------------------

In [13]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def translate(texts, max_length=64):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    # ✅ THIS LINE IS MANDATORY
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length
        )

    return tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )



In [14]:
preds = translate(dataset["source"][:200])
refs = dataset["target"][:200]

import sacrebleu
bleu = sacrebleu.corpus_bleu(preds, [refs])
print("mT5-small BLEU:", bleu.score)

mT5-small BLEU: 0.8223856922831985


In [15]:
import sacrebleu

bleu_before = sacrebleu.corpus_bleu(baseline_preds, [sample_ref])
bleu_after = sacrebleu.corpus_bleu(trained_preds, [sample_ref])

print("BLEU before fine-tuning:", bleu_before.score)
print("BLEU after fine-tuning :", bleu_after.score)


BLEU before fine-tuning: 0.018260469818849066
BLEU after fine-tuning : 0.523847308490564


In [16]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.translate.meteor_score import meteor_score

def compute_meteor(preds, refs):
    scores = []
    for p, r in zip(preds, refs):
        score = meteor_score([r.split()], p.split())
        scores.append(score)
    return sum(scores) / len(scores)


meteor_before = compute_meteor(baseline_preds, sample_ref)
meteor_after  = compute_meteor(trained_preds, sample_ref)

print("METEOR before fine-tuning:", meteor_before)
print("METEOR after fine-tuning :", meteor_after)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR before fine-tuning: 0.0
METEOR after fine-tuning : 0.040250589972705454


In [17]:
from rouge_score import rouge_scorer

def compute_rouge(preds, refs):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    r1 = r2 = rl = 0
    for p, r in zip(preds, refs):
        scores = scorer.score(r, p)
        r1 += scores['rouge1'].fmeasure
        r2 += scores['rouge2'].fmeasure
        rl += scores['rougeL'].fmeasure

    n = len(preds)
    return {
        "ROUGE-1": r1 / n,
        "ROUGE-2": r2 / n,
        "ROUGE-L": rl / n
    }


rouge_before = compute_rouge(baseline_preds, sample_ref)
rouge_after  = compute_rouge(trained_preds, sample_ref)

print("ROUGE before fine-tuning:", rouge_before)
print("ROUGE after fine-tuning :", rouge_after)


ROUGE before fine-tuning: {'ROUGE-1': 0.00019144144144144144, 'ROUGE-2': 0.0, 'ROUGE-L': 0.00019144144144144144}
ROUGE after fine-tuning : {'ROUGE-1': 0.1001714264367517, 'ROUGE-2': 0.013484511076079071, 'ROUGE-L': 0.08413577375585908}
